# Platform-agnostic training notebook

This notebook is designed to work on Kaggle, Google Colab, RunPod, and local GPU/CPU machines without depending on uv or Windows-specific commands.

In [8]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def resolve_repo_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "lux-llm-factory",
        cwd.parent / "lux-llm-factory",
        cwd / "LuxLLMFactory",
        cwd.parent / "LuxLLMFactory",
        Path("/kaggle/working/LuxLLMFactory"),
        Path("/kaggle/input/LuxLLMFactory"),
        Path("/kaggle/working/lux-llm-factory"),
        Path("/kaggle/input/lux-llm-factory"),
        Path("/kaggle/input/datasets/markzither/luxllmfactory"),
    ]

    for candidate in candidates:
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate

    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
            for candidate in sorted(base.glob("*")):
                if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
                    return candidate

    return cwd.parent if cwd.name == "notebooks" else cwd


REPO_DIR = resolve_repo_dir()
print("Working root:", Path.cwd().resolve())
print("Repo dir:", REPO_DIR)

if not REPO_DIR.exists():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        raise FileNotFoundError(
            "The repo was not found under the expected Kaggle locations. "
            "Please ensure your uploaded dataset folder contains pyproject.toml or .git and is visible in /kaggle/working or /kaggle/input."
        )
    raise FileNotFoundError("Repository directory not found. Please run this notebook from the repo root or upload the repo contents.")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and str(REPO_DIR).startswith("/kaggle/input/"):
    writable_repo = Path("/kaggle/working/lux-llm-factory")
    if not writable_repo.exists():
        shutil.copytree(REPO_DIR, writable_repo, dirs_exist_ok=True)
    REPO_DIR = writable_repo

os.chdir(REPO_DIR)
print("Repository ready at", REPO_DIR)

Working root: C:\Source\github\lod-mcp\lux-llm-factory
Repo dir: C:\Source\github\lod-mcp\lux-llm-factory
Repository ready at C:\Source\github\lod-mcp\lux-llm-factory


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

packages = [
    "pip",
    "torch>=2.11.0",
    "torchao>=0.16.0",
    "transformers>=4.44",
    "datasets>=2.20",
    "peft>=0.12",
    "trl>=0.10.1",
    "accelerate>=0.33",
    "evaluate>=0.4.1,<0.5",
    "pyyaml>=6",
    "sentencepiece>=0.2",
]

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio", "torchvision"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)

print("Dependencies installed.")

Dependencies installed.


In [10]:
import os
import platform

platform_name = "unknown"
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    platform_name = "kaggle"
elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    platform_name = "colab"
elif os.environ.get("RUNPOD_POD_ID"):
    platform_name = "runpod"
else:
    platform_name = "local"

print("Detected platform:", platform_name)
print("Python:", platform.python_version())

Detected platform: local
Python: 3.14.1


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR).resolve()
config_path = os.environ.get("TRAIN_CONFIG", "training/configs/first-run-cpu-mistral.yaml")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Using config:", config_path)

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    writable_root = Path("/kaggle/working") / "lux-llm-factory-output"
    writable_root.mkdir(parents=True, exist_ok=True)
    os.environ["TRAIN_OUTPUT_DIR"] = str(writable_root)
    print("Using writable output root:", writable_root)

    if not repo_dir.exists() or str(repo_dir).startswith("/kaggle/input/"):
        repo_dir = Path("/kaggle/working/lux-llm-factory").resolve()
        if not repo_dir.exists():
            repo_dir = Path(REPO_DIR).resolve()

os.chdir(repo_dir)
print("Training working directory:", repo_dir)

if not Path(config_path).is_absolute():
    config_path = str((repo_dir / config_path).resolve())

if os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])
    print("Hugging Face token detected from environment.")
elif os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    print("Hugging Face token detected from environment.")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        kaggle_token = user_secrets.get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = kaggle_token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = kaggle_token
        print("Hugging Face token detected from Kaggle secrets.")
    except Exception as exc:
        print("No Hugging Face token detected. On Windows, set it for future sessions with: setx HF_TOKEN \"your_token_here\"")
        print(f"Kaggle secret lookup failed: {exc}")
else:
    print("No Hugging Face token detected. On Windows, set it for future sessions with: setx HF_TOKEN \"your_token_here\"")


def run_streaming_command(command, cwd=None):
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


train_script = repo_dir / "scripts" / "train_first_sft.py"
run_streaming_command([sys.executable, str(train_script), "--config", config_path], cwd=str(repo_dir))

Using config: training/configs/first-run-cpu-mistral.yaml
$ c:\Source\github\lod-mcp\lux-llm-factory\.venv\Scripts\python.exe scripts/train_first_sft.py --config training/configs/first-run-cpu-mistral.yaml

